# NeuroFusionAI: Step 1 - Multimodal Data Preprocessing
This notebook walks through the data preparation and preprocessing pipeline of the three modalities:
1. **Hand-drawn Images** (resizing to 224x224 and splitting).
2. **Acoustic Voice Classification features** (feature standardization, duplicate removal, and stratified splitting).
3. **Tabular UPDRS Progression Telemonitoring features** (standardization and splitting).
We will import our modular preprocessing python package to execute the preprocessing.


In [1]:
import os
import sys
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import glob

# Add project root directory to path to enable local module imports
sys.path.append(os.path.abspath('..'))


## 1. Run Preprocessing Pipelines
We execute the preprocessing functions that resize images, standardize tabular features, handle missing data, and create stratified splits.


In [2]:
from preprocessing.image_preprocessing import split_and_preprocess_images
from preprocessing.voice_preprocessing import preprocess_voice
from preprocessing.telemonitor_preprocessing import preprocess_telemonitor

print('Running Image Preprocessing...')
split_and_preprocess_images('datasets/images', 'datasets/processed/images')

print('\nRunning Oxford Voice Preprocessing...')
preprocess_voice('datasets/voice/oxford/parkinsons.data', 'datasets')

print('\nRunning Telemonitoring Preprocessing...')
preprocess_telemonitor('datasets/voice/telemonitoring/parkinsons_updrs.data', 'datasets')


Running Image Preprocessing...

Running Oxford Voice Preprocessing...
[Oxford Voice Dataset] No missing values found.

Running Telemonitoring Preprocessing...
[Telemonitoring Dataset] No missing values found.


## 2. Visualize Preprocessed Drawings
Let's display some control (normal) vs Parkinson drawings to verify image processing.


In [3]:
normal_images = glob.glob('datasets/processed/images/train/normal/*.png')
parkinson_images = glob.glob('datasets/processed/images/train/parkinson/*.png')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
if normal_images:
    axes[0].imshow(Image.open(normal_images[0]))
    axes[0].set_title('Normal Control Drawing')
    axes[0].axis('off')
if parkinson_images:
    axes[1].imshow(Image.open(parkinson_images[0]))
    axes[1].set_title('Parkinson Drawing')
    axes[1].axis('off')
plt.show()


## 3. Verify Voice Acoustics Split Shape & Stats
Let's check the size and content of our voice splits.


In [4]:
train_voice = pd.read_csv('datasets/train/voice/oxford_train.csv')
print('Voice Train split shape:', train_voice.shape)
print('Distribution of Parkinson Status (1: Parkinson, 0: Healthy):')
print(train_voice['status'].value_counts(normalize=True))
train_voice.head(3)


Voice Train split shape: (136, 23)
Distribution of Parkinson Status (1: Parkinson, 0: Healthy):
status
1    0.757353
0    0.242647
Name: proportion, dtype: float64


## 4. Verify Telemonitoring UPDRS Split Shape & Stats
Let's inspect the tabular data and regression targets (`motor_UPDRS` and `total_UPDRS`) for severity tracking.


In [5]:
train_tele = pd.read_csv('datasets/train/telemonitoring/telemonitor_train.csv')
print('Telemonitoring Train split shape:', train_tele.shape)
print('Target statistics for severity assessment:')
print(train_tele[['motor_UPDRS', 'total_UPDRS']].describe())
train_tele.head(3)


Telemonitoring Train split shape: (4112, 21)
Target statistics for severity assessment:
       motor_UPDRS  total_UPDRS
count  4112.000000  4112.000000
mean     21.263284    29.004145
std       8.153072    10.740568
min       5.037700     7.000000
25%      14.890000    21.357500
50%      20.719000    27.490000
75%      27.555500    36.399000
max      39.511000    54.992000
